In [0]:
-- R2 Step 1: target table for policy chunks.
-- One row per ## section. Change Data Feed is required by the Delta Sync vector index.
CREATE TABLE IF NOT EXISTS claims.genai.policy_chunks (
  chunk_id    STRING NOT NULL COMMENT 'Stable id <filename>#<anchor>; anchor = lowercase heading, non-alnum -> single -, trimmed',
  doc_name    STRING COMMENT 'File name of the source policy document, e.g. 01_Claims_Process_Overview.md',
  heading     STRING COMMENT 'Section heading text (without the leading ##)',
  chunk_index INT COMMENT 'Position of the section within its document: 0, 1, 2, ...',
  text        STRING COMMENT 'Chunk text: title line + Applies-to line + blank + ## heading + blank + body',
  PRIMARY KEY (chunk_id) NOT ENFORCED
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT * FROM claims.genai.policy_chunks;

In [0]:
-- R2 Step 2: MERGE load #1 — chunk the 8 policy docs (one row per ## section) and upsert into policy_chunks.
-- Chunk id rule (Definitions): <filename>#<anchor>, anchor = lowercase heading, non-alnum runs -> single '-', trimmed.
-- Chunk text = title line + Applies-to line + blank + ## heading + blank + body.
-- The NOT MATCHED BY SOURCE clause drops sections removed from the binder on a reload.
MERGE INTO claims.genai.policy_chunks AS t
USING (
  SELECT
    concat(d.doc_name, '#',
           regexp_replace(regexp_replace(lower(split(p.part, '\n')[0]), '[^a-z0-9]+', '-'), '^-+|-+$', '')) AS chunk_id,
    d.doc_name,
    split(p.part, '\n')[0] AS heading,
    p.pos AS chunk_index,
    concat(split(d.parts[0], '\n')[0], '\n',
           filter(split(d.parts[0], '\n'), l -> l LIKE 'Applies to:%')[0], '\n\n## ', p.part) AS text
  FROM (
    SELECT regexp_extract(path, '/([^/]+\.md)$', 1) AS doc_name,
           split(CAST(content AS STRING), '\n## ') AS parts
    FROM read_files('/Volumes/claims/raw/landing/genai/policy_docs/', format => 'binaryFile')
  ) d
  LATERAL VIEW posexplode(slice(d.parts, 2, size(d.parts))) p AS pos, part
) AS s
ON t.chunk_id = s.chunk_id
WHEN MATCHED AND (t.doc_name <> s.doc_name OR t.heading <> s.heading
                  OR t.chunk_index <> s.chunk_index OR t.text <> s.text)
  THEN UPDATE SET *
WHEN NOT MATCHED
  THEN INSERT *
WHEN NOT MATCHED BY SOURCE
  THEN DELETE

In [0]:
-- R2 Step 3a: row counts and integrity of the chunk table.
SELECT
  count(*)                AS n_chunks,
  count(DISTINCT doc_name)  AS n_docs,
  count(DISTINCT chunk_id)  AS n_uniq_chunk_ids,
  min(chunk_index)          AS min_chunk_index,
  max(chunk_index)          AS max_chunk_index,
  sum(CASE WHEN chunk_id IS NULL OR doc_name IS NULL OR heading IS NULL
            OR chunk_index IS NULL OR text IS NULL THEN 1 ELSE 0 END) AS null_cells
FROM claims.genai.policy_chunks

In [0]:
-- R2 Step 3b: every expected chunk from gold_set.json must exist in the table with the exact id.
WITH gold AS (
  SELECT explode(questions) AS q
  FROM read_files('/Volumes/claims/raw/landing/genai/gold_set.json', format => 'json', multiLine => 'true')
),
refs AS (
  SELECT explode(from_json(q['expected_chunks'], 'array<string>')) AS expected_chunk
  FROM gold WHERE q['expected_chunks'] IS NOT NULL
),
expected AS (
  SELECT DISTINCT expected_chunk FROM refs WHERE expected_chunk IS NOT NULL
)
SELECT
  count(*)                                            AS n_expected_refs,
  sum(CASE WHEN c.chunk_id IS NOT NULL THEN 1 ELSE 0 END) AS found,
  sum(CASE WHEN c.chunk_id IS NULL THEN 1 ELSE 0 END)     AS missing
FROM expected e
LEFT JOIN claims.genai.policy_chunks c
  ON c.chunk_id = e.expected_chunk

In [0]:
-- R2 Step 3c: sample chunk text (layout: title line, Applies-to line, blank, ## heading, blank, body).
SELECT chunk_id, doc_name, heading, chunk_index, text
FROM claims.genai.policy_chunks


In [0]:
-- R2 Step 4: MERGE load #2 — re-run of the same unchanged binder.
-- A reload of an unchanged binder must change nothing: 0 inserted, 0 updated, 0 deleted.
MERGE INTO claims.genai.policy_chunks AS t
USING (
  SELECT
    concat(d.doc_name, '#',
           regexp_replace(regexp_replace(lower(split(p.part, '\n')[0]), '[^a-z0-9]+', '-'), '^-+|-+$', '')) AS chunk_id,
    d.doc_name,
    split(p.part, '\n')[0] AS heading,
    p.pos AS chunk_index,
    concat(split(d.parts[0], '\n')[0], '\n',
           filter(split(d.parts[0], '\n'), l -> l LIKE 'Applies to:%')[0], '\n\n## ', p.part) AS text
  FROM (
    SELECT regexp_extract(path, '/([^/]+\.md)$', 1) AS doc_name,
           split(CAST(content AS STRING), '\n## ') AS parts
    FROM read_files('/Volumes/claims/raw/landing/genai/policy_docs/', format => 'binaryFile')
  ) d
  LATERAL VIEW posexplode(slice(d.parts, 2, size(d.parts))) p AS pos, part
) AS s
ON t.chunk_id = s.chunk_id
WHEN MATCHED AND (t.doc_name <> s.doc_name OR t.heading <> s.heading
                  OR t.chunk_index <> s.chunk_index OR t.text <> s.text)
  THEN UPDATE SET *
WHEN NOT MATCHED
  THEN INSERT *
WHEN NOT MATCHED BY SOURCE
  THEN DELETE

In [0]:
-- R2 Step 5: table history — must show at least two MERGE loads; the last one with 0 inserted and 0 deleted.
DESCRIBE HISTORY claims.genai.policy_chunks

In [0]:
%python
# R2 Step 6: Vector Search — endpoint claims_vs + Delta Sync index on policy_chunks.
# Endpoint: STANDARD. Index: source claims.genai.policy_chunks, PK chunk_id, embedding
# source column text, model databricks-gte-large-en, pipeline type TRIGGERED.
# A TRIGGERED pipeline follows the table automatically on each sync call — a binder
# reload (update/new/drop of sections) is picked up by re-running the sync.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import vectorsearch
import time

w = WorkspaceClient()

# --- endpoint: create only if missing ---
try:
    ep = w.vector_search_endpoints.get_endpoint(name="claims_vs")
    print(f"Endpoint claims_vs exists, state={ep.endpoint_status.state}")
except Exception:
    w.vector_search_endpoints.create_endpoint_and_wait(
        name="claims_vs", endpoint_type=vectorsearch.EndpointType.STANDARD)
    print("Endpoint claims_vs created")

# --- index: create only if missing ---
INDEX = "claims.genai.policy_chunks_idx"
try:
    idx = w.vector_search_indexes.get_index(index_name=INDEX)
    print(f"Index {INDEX} exists, state={idx.status.detailed_state}")
except Exception:
    idx = w.vector_search_indexes.create_index(
        name=INDEX,
        endpoint_name="claims_vs",
        primary_key="chunk_id",
        index_type=vectorsearch.VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=vectorsearch.DeltaSyncVectorIndexSpecRequest(
            source_table="claims.genai.policy_chunks",
            embedding_source_columns=[vectorsearch.EmbeddingSourceColumn(
                name="text",
                embedding_model_endpoint_name="databricks-gte-large-en")],
            pipeline_type=vectorsearch.PipelineType.TRIGGERED))

# --- wait until ready, then sync and verify every chunk is indexed ---
w.vector_search_indexes.sync_index(index_name=INDEX)
for _ in range(120):
    idx = w.vector_search_indexes.get_index(index_name=INDEX)
    if idx.status.ready:
        break
    time.sleep(10)
print(f"ready={idx.status.ready}, state={idx.status.detailed_state}, "
      f"indexed_rows={idx.status.indexed_row_count}")

sample = w.vector_search_indexes.query_index(
    index_name=INDEX, columns=["chunk_id", "doc_name", "heading"],
    query_text="What is the waiting period for a new policy?", num_results=3)
for row in sample.result.data_array:
    print(row)